# Support Ticket Router

TaskNest support tickets are classified by an OpenAI model and routed to Billing, Technical Support, or General Support. Add `OPENAI_API_KEY` to Colab Secrets, grant this notebook access, and run the cells in order.

## Setup

Install the dependencies, load the API key from Colab Secrets, and configure the classifier model.

In [ ]:
!pip -q install langgraph langchain-openai

In [ ]:
import os
from typing import Literal, NotRequired, TypedDict

from google.colab import userdata
from IPython.display import Image, display
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

MODEL_NAME = "gpt-4.1-mini"
llm = ChatOpenAI(model=MODEL_NAME, temperature=0, max_retries=0)

## State and nodes

The classifier selects one category. Each team node adds a fixed reply and a team name.

In [ ]:
Category = Literal["billing", "technical", "general"]


class TicketState(TypedDict):
    ticket: str
    category: NotRequired[Category]
    team: NotRequired[str]
    reply: NotRequired[str]


class TicketClassification(BaseModel):
    category: Category


CLASSIFICATION_INSTRUCTIONS = """You are a customer support ticket classifier for TaskNest, a to-do app.

Read the customer's ticket and classify it into one of three categories:

- billing: Payments, invoices, refunds, and subscriptions.
- technical: Bugs, crashes, error messages, and login problems.
- general: Feature requests, general questions, thank-you messages, and anything else.

Choose the category based on the main problem described in the ticket, not just keywords.

If a ticket mentions more than one topic, choose the category that best matches the main issue.

Return only the category."""

classifier = llm.with_structured_output(TicketClassification)


def classify(state: TicketState) -> dict[str, Category]:
    result = classifier.invoke(
        [
            ("system", CLASSIFICATION_INSTRUCTIONS),
            ("human", state["ticket"]),
        ]
    )
    return {"category": result.category}


def billing_team(state: TicketState) -> dict[str, str]:
    return {
        "team": "Billing",
        "reply": "Your ticket has been sent to our Billing team.",
    }


def tech_team(state: TicketState) -> dict[str, str]:
    return {
        "team": "Technical Support",
        "reply": "Your ticket has been sent to our Technical Support team.",
    }


def general_team(state: TicketState) -> dict[str, str]:
    return {
        "team": "General Support",
        "reply": "Your ticket has been sent to our General Support team.",
    }

## Build and display the graph

The conditional edge sends the ticket to the team for its category. The graph ends after that team adds its reply.

In [ ]:
def route(state: TicketState) -> Category:
    return state["category"]


builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing_team", billing_team)
builder.add_node("tech_team", tech_team)
builder.add_node("general_team", general_team)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route,
    {
        "billing": "billing_team",
        "technical": "tech_team",
        "general": "general_team",
    },
)
builder.add_edge("billing_team", END)
builder.add_edge("tech_team", END)
builder.add_edge("general_team", END)

graph = builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Try sample tickets

Each ticket invokes the graph once. Compare the predicted category with the expected category, especially when a ticket mentions more than one topic.

In [ ]:
test_tickets = [
    ("I see two TaskNest charges for the same annual renewal. Can you reverse one?", "billing"),
    ("The app works fine. Where can I download a receipt for my subscription?", "billing"),
    ("The app closes every time I tap Pay. I want help fixing the crash so I can finish checking out.", "technical"),
    ("I reset my password, but signing in still sends me back to the login screen.", "technical"),
    ("Could you add a way to group my tasks by project and collapse each group?", "general"),
    ("Thanks for fixing the login error. Everything works again, and I just wanted to say thanks!", "general"),
]

for ticket, expected_category in test_tickets:
    result = graph.invoke({"ticket": ticket})
    print(f"Ticket: {result['ticket']}")
    print(f"Category: {result['category']} (expected: {expected_category})")
    print(f"Team: {result['team']}")
    print(f"Reply: {result['reply']}")
    print()

# BONUS – Interactive Multilingual Support Assistant

Run the two cells below after the original graph has been compiled. Enter a ticket in English or Hebrew, then select **Route Ticket**. **Clear** resets the form and the result.

In [ ]:
import ipywidgets as widgets
import re
from html import escape

from google.colab import output
from IPython.display import display

output.enable_custom_widget_manager()

In [ ]:
TASKNEST_HEBREW_LETTERS = re.compile(r"[א-תװ-ײיִ-ﭏ]")

TASKNEST_HEBREW_REPLIES = {
    "Billing": "הפנייה שלך הועברה לצוות החיובים שלנו.",
    "Technical Support": "הפנייה שלך הועברה לצוות התמיכה הטכנית שלנו.",
    "General Support": "הפנייה שלך הועברה לצוות התמיכה הכללית שלנו.",
}


def detect_ticket_language(ticket_text):
    return "Hebrew" if TASKNEST_HEBREW_LETTERS.search(ticket_text) else "English"


TASKNEST_STYLES = """
<style>
.tasknest-assistant {
    box-sizing: border-box; margin: 18px auto; padding: clamp(16px, 4vw, 32px);
    background: #fff; border: 1px solid #dbe4ef; border-radius: 18px;
    box-shadow: 0 8px 26px rgba(31, 59, 98, .08); color: #142541;
    gap: 12px; font-family: system-ui, sans-serif;
}
.tasknest-assistant h2 { margin: 0; font: 700 clamp(21px, 4vw, 29px)/1.2 system-ui, sans-serif; }
.tasknest-assistant .tasknest-subtitle { margin: 6px 0 8px; color: #50627c; font-size: 14px; }
.tasknest-assistant .tasknest-label { font-size: 14px; font-weight: 650; color: #142541; }
.tasknest-assistant textarea {
    box-sizing: border-box; padding: 12px; min-height: 135px;
    font: 14px/1.55 system-ui, sans-serif; color: #142541; background: #f8fafc;
    border: 1px solid #dbe4ef; border-radius: 10px;
}
.tasknest-assistant textarea:focus { outline: 2px solid #72a6fa; outline-offset: 1px; }
.tasknest-assistant button, .tasknest-assistant button:hover {
    font: 650 14px system-ui, sans-serif; border-radius: 8px; box-shadow: none;
}
.tasknest-assistant .tasknest-route { background: #225bc8; color: #fff; }
.tasknest-assistant .tasknest-route:hover:not(:disabled) { background: #184aa9; color: #fff; }
.tasknest-assistant .tasknest-clear { background: #f1f5fa; color: #233c5e; border: 1px solid #dbe4ef; }
.tasknest-assistant .tasknest-message { color: #405779; font-size: 14px; }
.tasknest-assistant .tasknest-result {
    background: #f3f7ff; border: 1px solid #d7e5fb; border-radius: 12px;
    padding: 16px; font: 14px/1.5 system-ui, sans-serif; color: #142541;
}
.tasknest-assistant .tasknest-details {
    display: flex; flex-wrap: wrap; gap: 12px 32px; margin: 0 0 14px;
}
.tasknest-assistant dt, .tasknest-assistant .tasknest-reply-label {
    font-size: 13px; font-weight: 650; color: #405779;
}
.tasknest-assistant dd { margin: 3px 0 0; font-weight: 650; }
.tasknest-assistant .tasknest-reply {
    margin: 5px 0 0; padding: 10px 12px; background: #fff;
    border: 1px solid #dbe4ef; border-radius: 8px; overflow-wrap: anywhere;
}
</style>
"""


def show_tasknest_assistant():
    header = widgets.HTML(TASKNEST_STYLES + """
        <h2>TaskNest Support Assistant</h2>
        <p class="tasknest-subtitle">Interactive Multilingual Ticket Router</p>
    """)
    label = widgets.HTML('<div class="tasknest-label">Customer Ticket</div>')
    ticket = widgets.Textarea(
        value="",
        description="Customer Ticket",
        placeholder="Describe how we can help...",
        layout=widgets.Layout(width="100%", height="145px"),
        style={"description_width": "0px"},
    )
    # Keep the widget description available while using the label above the field.
    ticket.add_class("tasknest-ticket")
    header.value += """
        <style>
        .tasknest-assistant .tasknest-ticket label {
            position: absolute; clip-path: inset(50%); width: 1px; height: 1px;
            overflow: hidden; white-space: nowrap;
        }
        </style>
    """
    route_button = widgets.Button(description="Route Ticket", layout=widgets.Layout(width="132px"))
    clear_button = widgets.Button(description="Clear", layout=widgets.Layout(width="84px"))
    route_button.add_class("tasknest-route")
    clear_button.add_class("tasknest-clear")
    actions = widgets.HBox([route_button, clear_button], layout=widgets.Layout(flex_flow="row wrap", gap="8px"))
    result = widgets.HTML(layout=widgets.Layout(display="none"))
    panel = widgets.VBox(
        [header, label, ticket, actions, result],
        layout=widgets.Layout(width="100%", max_width="680px"),
    )
    panel.add_class("tasknest-assistant")

    def show_result(markup):
        result.value = markup
        result.layout.display = "" if markup else "none"

    def show_message(message):
        show_result(f'<div class="tasknest-message" role="status">{escape(message)}</div>')

    def handle_clear(_):
        if not route_button.disabled:
            ticket.value = ""
            show_result("")

    def handle_route(_):
        if route_button.disabled:
            return
        ticket_text = ticket.value
        if not ticket_text.strip():
            show_message("Enter a customer ticket before routing.")
            return

        language = detect_ticket_language(ticket_text)
        route_button.disabled = clear_button.disabled = ticket.disabled = True
        show_message("Processing ticket...")
        try:
            routed = graph.invoke({"ticket": ticket_text})
            team, _category, english_reply = routed["team"], routed["category"], routed["reply"]
            if language == "Hebrew":
                reply = TASKNEST_HEBREW_REPLIES[team]
                direction, locale, alignment = "rtl", "he", "right"
            else:
                reply = english_reply
                direction, locale, alignment = "ltr", "en", "left"
            show_result(f"""
                <section class="tasknest-result" aria-label="Routing result" aria-live="polite">
                    <dl class="tasknest-details">
                        <div><dt>Detected Language</dt><dd>{language}</dd></div>
                        <div><dt>Assigned Team</dt><dd>{escape(team)}</dd></div>
                    </dl>
                    <div class="tasknest-reply-label">Customer Reply</div>
                    <p class="tasknest-reply" lang="{locale}" dir="{direction}"
                       style="text-align: {alignment}">{escape(reply)}</p>
                </section>
            """)
        except Exception:
            show_message("We couldn't route your ticket. Please try again.")
        finally:
            route_button.disabled = clear_button.disabled = ticket.disabled = False

    def dispose():
        route_button.on_click(handle_route, remove=True)
        clear_button.on_click(handle_clear, remove=True)
        for widget in (panel, actions, header, label, ticket, route_button, clear_button, result):
            widget.close()

    route_button.on_click(handle_route)
    clear_button.on_click(handle_clear)
    display(panel)
    return dispose


if "dispose_tasknest_assistant" in globals():
    dispose_tasknest_assistant()
dispose_tasknest_assistant = show_tasknest_assistant()